# 3 Modeling
Loads the cleaned data saved by `data_cleaning.ipynb`.

In [10]:
import numpy as np
import pandas as pd
from carpricepredictor.shared import (
    load_split, num_cols, cat_cols, cat_features, BEST_PARAMS_PATH, CLEAN_CSV, FEATURES_CSV,
)

# After eda feature engineering 
- tranforming year into age 
- correlating odometer and age (how much was car driven) 
- loging odometer so the change curve is closer to a stright line 
- cylinders to int wiht other as nan as elec as 0
- is classic for older cars (older that 1995)
- merging truck and pickups are the same

The iteration 2 listing fields (`is_dealer`, `has_vin`, `desc_len`, `lat` / `long`, keyword flags) are built in `data_cleaning.ipynb`, because they need the raw listing text. The cell below adds the rest and saves `data/vehicles_features.csv`, which `final_model.ipynb` reads through `load_split()`.

In [11]:
def add_features(df):
    df = df.copy()
    df["age"] = (2021 - df["year"]).clip(lower=0)
    df["miles_per_year"] = df["odometer"] / df["age"].clip(lower=1)
    df["log_odometer"] = np.log1p(df["odometer"])
    df["is_classic"] = (df["year"] < 1995).astype(int)

    # cylinders as a number: electric = 0, diff engines like rotary = NaN (we handle in the pipeline)
    df["cylinders_num"] = df["cylinders"].str.extract(r"(\d+)")[0].astype(float)
    df.loc[df["fuel"] == "electric", "cylinders_num"] = 0

    df["type"] = df["type"].replace({"truck": "pickup"})
    return df.drop(columns=["cylinders", "year"])

add_features(pd.read_csv(CLEAN_CSV)).to_csv(FEATURES_CSV, index=False)

## Spliting the data
`load_split()` reads `data/vehicles_features.csv`: `y = log1p(price)`, 80/20 split with `random_state=42` (same split in `final_model.ipynb`).

In [12]:
X_train, X_test, y_train, y_test = load_split()
X_train.shape, X_test.shape

((164308, 32), (41077, 32))

# ⚠️⚠️⚠️ QUICK FLAG ⚠️⚠️⚠️

**`QUICK = True`** → models train on a **40k-row sample** of the training set (fast experiments).

The final model is trained on the **full** training set in `final_model.ipynb`, using the params this notebook saves to `data/best_params.json`.

In [13]:
QUICK = False


In [14]:
if QUICK:
    X_tr = X_train.sample(40_000, random_state=42)
    y_tr = y_train.loc[X_tr.index]
else:
    X_tr, y_tr = X_train, y_train

# Training models to test
- DummyRegressor(baseline)
- LinearRegression 
- RandomForestRegressor
- Ridge
- HitsGradientBoostingRegressor
- CatBoost


In [15]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from catboost import CatBoostRegressor

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# raw "odometer" is left out on purpose: log_odometer replaces it
preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num_cols),
    ("model", OneHotEncoder(min_frequency=50, handle_unknown="infrequent_if_exist", sparse_output=False), ["model"]),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

In [17]:
regressors = [
    DummyRegressor(strategy="median"),
    LinearRegression(),
    Ridge(),
    DecisionTreeRegressor(random_state=42),
    RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42),
    HistGradientBoostingRegressor(random_state=42),
    KNeighborsRegressor(n_jobs=-1),
    CatBoostRegressor(verbose=0, random_state=42),
]


In [18]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

results = []
for reg in regressors:
    pipe = make_pipeline(preprocess, reg)
    scores = cross_validate(
        pipe, X_tr, y_tr, cv=cv,
        scoring=["neg_root_mean_squared_error", "neg_mean_absolute_error", "r2"],
        n_jobs=-1,
    )
    results.append({
        "model": type(reg).__name__,
        "rmse_log": -scores["test_neg_root_mean_squared_error"].mean(),
        "mae_log": -scores["test_neg_mean_absolute_error"].mean(),
        "r2": scores["test_r2"].mean(),
    })

results = pd.DataFrame(results).sort_values("rmse_log").reset_index(drop=True)
results

,model,rmse_log,mae_log,r2
0,CatBoostRegressor,0.338422,0.213232,0.850245
1,HistGradientBoostingRegressor,0.357840,0.231266,0.832568
2,RandomForestRegressor,0.363325,0.230392,0.827380
3,Ridge,0.413085,0.273508,0.776885
4,LinearRegression,0.413103,0.273473,0.776866
5,KNeighborsRegressor,0.461951,0.331319,0.720976
6,DecisionTreeRegressor,0.510314,0.327116,0.659502
7,DummyRegressor,0.875559,0.707415,-0.002325


# Hyperparameter tuning (top 3)
Top 3 from CV: **CatBoost**, **HistGradientBoosting**, **RandomForest**.

Baseline to beat: **LinearRegression** (DummyRegressor is just the sanity floor).

`RandomizedSearchCV` with the same 5-fold `cv`, scored on RMSE of log-price.

In [19]:
from scipy.stats import loguniform, randint
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV

In [20]:
# models run single-threaded here, the search itself runs folds in parallel
search_spaces = {
    "CatBoostRegressor": (
        CatBoostRegressor(verbose=0, random_state=42, thread_count=1),
        {
            "reg__depth": [4, 6, 8, 10],
            "reg__learning_rate": loguniform(0.02, 0.3),
            "reg__iterations": [500, 1000, 1500],
            "reg__l2_leaf_reg": loguniform(1, 10),
        },
        20,
    ),
    "HistGradientBoostingRegressor": (
        HistGradientBoostingRegressor(random_state=42),
        {
            "reg__learning_rate": loguniform(0.02, 0.3),
            "reg__max_iter": [200, 500, 1000],
            "reg__max_leaf_nodes": randint(15, 128),
            "reg__min_samples_leaf": randint(10, 100),
            "reg__l2_regularization": loguniform(1e-3, 10),
        },
        20,
    ),
    "RandomForestRegressor": (
        RandomForestRegressor(random_state=42, n_jobs=1),
        {
            "reg__n_estimators": [200, 400],
            "reg__max_features": [0.2, 0.33, 0.5, "sqrt"],
            "reg__min_samples_leaf": [1, 2, 4, 8],
            "reg__max_depth": [None, 20, 30],
        },
        10,  # RF is the slowest, fewer tries
    ),
}

In [21]:
searches = {}
for name, (reg, params, n_iter) in search_spaces.items():
    search = RandomizedSearchCV(
        Pipeline([("prep", preprocess), ("reg", reg)]),
        params, n_iter=n_iter, cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1, random_state=42,
    )
    search.fit(X_tr, y_tr)
    searches[name] = search
    print(name, round(-search.best_score_, 4), search.best_params_)

CatBoostRegressor 0.3128 {'reg__depth': 10, 'reg__iterations': 1500, 'reg__l2_leaf_reg': np.float64(2.858051065806936), 'reg__learning_rate': np.float64(0.16767510257004176)}
HistGradientBoostingRegressor 0.3235 {'reg__l2_regularization': np.float64(0.029204338471814112), 'reg__learning_rate': np.float64(0.06877172495876586), 'reg__max_iter': 1000, 'reg__max_leaf_nodes': 76, 'reg__min_samples_leaf': 60}
RandomForestRegressor 0.3517 {'reg__n_estimators': 200, 'reg__min_samples_leaf': 1, 'reg__max_features': 0.2, 'reg__max_depth': None}


In [22]:
baseline = results.set_index("model").loc["LinearRegression", "rmse_log"]

tuned = pd.DataFrame({
    "model": list(searches),
    "rmse_default": [results.set_index("model").loc[n, "rmse_log"] for n in searches],
    "rmse_tuned": [-s.best_score_ for s in searches.values()],
}).sort_values("rmse_tuned").reset_index(drop=True)
tuned["gain_vs_default"] = tuned["rmse_default"] - tuned["rmse_tuned"]
tuned["gain_vs_linear"] = baseline - tuned["rmse_tuned"]
tuned

,model,rmse_default,rmse_tuned,gain_vs_default,gain_vs_linear
0,CatBoostRegressor,0.338422,0.312849,0.025573,0.100253
1,HistGradientBoostingRegressor,0.357840,0.323463,0.034377,0.089639
2,RandomForestRegressor,0.363325,0.351705,0.011620,0.061398


# CatBoost deep tuning
CatBoost won (0.3632). Two steps:
1. **One-hot vs native categories**: CatBoost can use the raw category columns directly (its own target encoding), often better for high-cardinality columns like `model`.
2. **Narrow search** on the better version. Last search hit the edges (`iterations=1500`, `l2_leaf_reg≈9.6`), so this one goes past them.

In [23]:
def to_catboost(X):
    """Raw categories for CatBoost's native handling (cat_features goes in fit())."""
    X = X[num_cols + cat_features].copy()
    X[cat_features] = X[cat_features].fillna("missing").astype(str)
    return X

X_tr_cb = to_catboost(X_tr)

In [24]:
best = {k.removeprefix("reg__"): v for k, v in searches["CatBoostRegressor"].best_params_.items()}
native = CatBoostRegressor(**best, verbose=0, random_state=42, thread_count=1)

# cat_features goes in fit(), sklearn can't clone it from the constructor
scores = cross_validate(native, X_tr_cb, y_tr, cv=cv, scoring="neg_root_mean_squared_error",
                        n_jobs=-1, params={"cat_features": cat_features})
onehot_rmse = -searches["CatBoostRegressor"].best_score_
native_rmse = -scores["test_score"].mean()

USE_NATIVE = native_rmse < onehot_rmse
print("one-hot:", round(onehot_rmse, 4), "| native:", round(native_rmse, 4), "| using", "native" if USE_NATIVE else "one-hot")

one-hot: 0.3128 | native: 0.3135 | using one-hot


In [25]:
params = {
    "depth": [7, 8, 9, 10],
    "learning_rate": loguniform(0.03, 0.1),
    "iterations": [1500, 2000, 3000],
    "l2_leaf_reg": loguniform(5, 30),
    "random_strength": loguniform(0.5, 5),
}

if USE_NATIVE:
    est = CatBoostRegressor(verbose=0, random_state=42, thread_count=1)
    X_search, fit_params = X_tr_cb, {"cat_features": cat_features}
else:
    est = Pipeline([("prep", preprocess), ("reg", CatBoostRegressor(verbose=0, random_state=42, thread_count=1))])
    X_search, fit_params = X_tr, {}
    params = {f"reg__{k}": v for k, v in params.items()}

cat_search = RandomizedSearchCV(
    est, params, n_iter=20, cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1, random_state=42,
)
cat_search.fit(X_search, y_tr, **fit_params)
print(round(-cat_search.best_score_, 4), cat_search.best_params_)

0.3105 {'reg__depth': 10, 'reg__iterations': 3000, 'reg__l2_leaf_reg': np.float64(5.187855301194419), 'reg__learning_rate': np.float64(0.0964420650918313), 'reg__random_strength': np.float64(3.3994812107955625)}


In [26]:
pd.DataFrame({
    "step": ["LinearRegression (baseline)", "CatBoost default", "CatBoost tuned (round 1)",
             "CatBoost native, round-1 params", "CatBoost tuned (round 2)"],
    "rmse_log": [baseline, results.set_index("model").loc["CatBoostRegressor", "rmse_log"],
                 onehot_rmse, native_rmse, -cat_search.best_score_],
})

,step,rmse_log
0,LinearRegression (baseline),0.413103
1,CatBoost default,0.338422
2,CatBoost tuned (round 1),0.312849
3,"CatBoost native, round-1 params",0.313486
4,CatBoost tuned (round 2),0.310499


# Save the best params
Written to `data/best_params.json`; `final_model.ipynb` reads it. Re-running this notebook overwrites it.

In [27]:
import json

best_params = {
    "model": "CatBoostRegressor",
    "use_native": bool(USE_NATIVE),
    "params": {k.removeprefix("reg__"): v.item() if hasattr(v, "item") else v
               for k, v in cat_search.best_params_.items()},
    "cv_rmse_log": round(-cat_search.best_score_, 4),
    "quick": QUICK,
    "n_train_rows": len(X_tr),
}
BEST_PARAMS_PATH.write_text(json.dumps(best_params, indent=2))
best_params

{'model': 'CatBoostRegressor',
 'use_native': False,
 'params': {'depth': 10,
  'iterations': 3000,
  'l2_leaf_reg': 5.187855301194419,
  'learning_rate': 0.0964420650918313,
  'random_strength': 3.3994812107955625},
 'cv_rmse_log': np.float64(0.3105),
 'quick': False,
 'n_train_rows': 164308}